In [ ]:
import pandas as pd
from pprint import pprint

# ======================================================
# Configuración
# ======================================================

EXCEL_FILE = "chip_layout_normalized.xlsx"

El archivo excel se compone de 4 hojas Metadata, Devices, Ports, Measurements

In [ ]:
# ======================================================
# Leer el Excel
# ======================================================

metadata_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Metadata"
)

devices_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Devices"
)

ports_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Ports"
)

measurements_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Measurements"
)

Iterrows entrega el índice de la fila y la fila completa, este es un ejemplo de lo que sucedería si se usasen ambas variables

In [ ]:
for indice, row in devices_df.iterrows():
    print(indice, row["Device"], row["Type"], row["Description"], row["Measure"])

In [14]:
# ======================================================
# Construir el objeto CHIP
# ======================================================

chip = {}

# ------------------------------------------------------
# Añadir los dispositivos
# ------------------------------------------------------

for _, row in devices_df.iterrows():

    chip[row["Device"]] = {

        "type": row["Type"],

        "measure": row["Measure"],

        "ports": {} # Es un diccionario vacío que se llenará más adelante con los puertos del dispositivo

    }

In [15]:
# ------------------------------------------------------
# Añadir los puertos a cada dispositivo
# ------------------------------------------------------

for _, row in ports_df.iterrows():
    # Chequea cada una de las filas del DataFrame de puertos y añade la información 
    # correspondiente al diccionario del dispositivo correspondiente en el objeto CHIP.

    device = row["Device"] # Esto es valido debido a que cada puerto tiene un dispositivo 
    #asociado en la columna "Device" que coincide con el nombre del dispositivo en el DataFrame de devices.

    chip[device]["ports"][row["Port"]] = {

        "x": row["X (um)"],

        "y": row["Y (um)"]

    }

In [16]:
# ======================================================
# Mostrar la estructura cargada
# ======================================================

print("=" * 80)
print("CHIP CARGADO")
print("=" * 80)

pprint(chip)

CHIP CARGADO
{'BTS_01': {'measure': 'YES',
            'ports': {'P1': {'x': 952.5, 'y': 300},
                      'P10': {'x': 1524.0, 'y': 300},
                      'P11': {'x': 1587.5, 'y': 300},
                      'P12': {'x': 1651.0, 'y': 300},
                      'P13': {'x': 1714.5, 'y': 300},
                      'P14': {'x': 1778.0, 'y': 300},
                      'P15': {'x': 1841.5, 'y': 300},
                      'P16': {'x': 1905.0, 'y': 300},
                      'P17': {'x': 1968.5, 'y': 300},
                      'P2': {'x': 1016.0, 'y': 300},
                      'P3': {'x': 1079.5, 'y': 300},
                      'P4': {'x': 1143.0, 'y': 300},
                      'P5': {'x': 1206.5, 'y': 300},
                      'P6': {'x': 1270.0, 'y': 300},
                      'P7': {'x': 1333.5, 'y': 300},
                      'P8': {'x': 1397.0, 'y': 300},
                      'P9': {'x': 1460.5, 'y': 300}},
            'type': 'BTS1x16'},
 'MMI_01': {'mea

In [22]:
measurement_plan = []

# ======================================================
# Construir la lista de mediciones
# ======================================================
for _, row in measurements_df.iterrows():

    measurement = {

        "device": row["Device"],

        "input": row["Input"],

        "output": row["Output"],

        "enabled": row["Enabled"] == "YES"

        # "status": "PENDING",

        # "power": None,

        # "timestamp": None

    }

    measurement_plan.append(measurement)

In [23]:
# ======================================================
# Mostrar el plan de medidas
# ======================================================

print("=" * 80)
print("PLAN DE MEDIDAS")
print("=" * 80)

pprint(measurement_plan)

PLAN DE MEDIDAS
[{'device': 'MZI_01', 'enabled': False, 'input': 'P1', 'output': 'P2'},
 {'device': 'MMI_01', 'enabled': False, 'input': 'P1', 'output': 'P2'},
 {'device': 'MMI_01', 'enabled': False, 'input': 'P1', 'output': 'P3'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P2'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P3'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P4'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P5'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P6'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P7'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P8'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P9'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P10'},
 {'device': 'BTS_01', 'enabled': False, 'input': 'P1', 'output': 'P11'},
 {'device': 'BTS_01', 'enabled': False, 'input

In [27]:
device = chip["MMI_01"]
device

{'type': 'MMI1x2',
 'measure': 'NO',
 'ports': {'P1': {'x': 762.0, 'y': 200},
  'P2': {'x': 825.5, 'y': 200},
  'P3': {'x': 889.0, 'y': 200}}}

In [30]:
for measurement in measurement_plan:

    # ¿Esta medición está habilitada?
    if not measurement["enabled"]:
        continue

    device = chip[measurement["device"]]

    input_port = device["ports"][measurement["input"]]

    output_port = device["ports"][measurement["output"]]

    print("=" * 50)

    print(f"Dispositivo : {measurement['device']}")
    print(f"Entrada     : {measurement['input']}")
    print(f"Salida      : {measurement['output']}")

    print(
        f"Mover fibra izquierda -> "
        f"({input_port['x']}, {input_port['y']})"
    )

    print(
        f"Mover fibra derecha   -> "
        f"({output_port['x']}, {output_port['y']})"
    )

    # --------------------------------------------------
    # Rutinas de posicionamiento y medición
    # --------------------------------------------------

Dispositivo : MMI_02
Entrada     : P1
Salida      : P3
Mover fibra izquierda -> (2635.0, 100)
Mover fibra derecha   -> (2762.0, 100)
Dispositivo : MMI_02
Entrada     : P1
Salida      : P4
Mover fibra izquierda -> (2635.0, 100)
Mover fibra derecha   -> (2825.5, 100)
Dispositivo : MMI_02
Entrada     : P2
Salida      : P3
Mover fibra izquierda -> (2698.5, 100)
Mover fibra derecha   -> (2762.0, 100)
Dispositivo : MMI_02
Entrada     : P2
Salida      : P4
Mover fibra izquierda -> (2698.5, 100)
Mover fibra derecha   -> (2825.5, 100)
